# DQN Backtesting

<div class="alert alert-success">In this notebook, I backtest the DQN model performance.</div> 

Import all necessary libraries

In [ ]:
import os
import time
import datetime
import json
from pathlib import Path
from sklearn.utils import shuffle
import pyarrow.parquet as pq
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Optional


In [ ]:
def convert_data_to_state(file_path: str, window: int = 60) -> tuple[np.ndarray, np.ndarray]:
    features = ['RSI_norm', 'MACD_line_norm', 'MACD_signal_norm', 'Volume_norm', 'Return']
    X = []
    metadata = []
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        if len(data) < 100: # minimum 40 days of data (applicable to validation data)
            return None, None
        for i in range(len(data) - window + 1): 
            X.append(data[features].iloc[i:i+window].values)
            metadata.append({
                'target': data['Target'].iloc[i+window-1], 
                'date': data.index[i+window-1], 
                'close': data['Close'].iloc[i+window-1]})
        return np.array(X), np.array(metadata)
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return None, None

In [ ]:
file_path = "dataset_train/AAPL"

In [ ]:
print("#" * 40 + " -Start- "+"#" * 40)
print("\n", chr(128994),"The ", file_path, " is selected for testing the data loading...")
state, metadata = convert_data_to_state(file_path) # read file content and convert them
assert state is not None and metadata is not None, "State conversion failed."
print("\n", chr(128992), "This file has ", state.shape[0], " days of observations.")
assert state.shape[0] == len(metadata), "Mismatch between features and metadata length."

source_data = pd.read_parquet(file_path, engine='pyarrow') # (n rows, 8 features)
obs_index = state.shape[0] - 1
source_index = source_data.shape[0] - 1 
assert source_data.iloc[source_index]['RSI_norm'] == state[obs_index][-1][0], "RSI mismatch."
assert source_data.iloc[source_index]['MACD_line_norm'] == state[obs_index][-1][1], "MACD_line mismatch."
assert source_data.iloc[source_index]['MACD_signal_norm'] == state[obs_index][-1][2], "MACD_signal mismatch."
assert source_data.iloc[source_index]['Volume_norm'] == state[obs_index][-1][3], "Volume mismatch."
assert source_data.iloc[source_index]['Return'] == state[obs_index][-1][4], "Return mismatch."
print("\n", chr(128640), "Test passed")



indicator = "Indicator"
print("\n🍋 Source's data shape:", source_data.shape if source_data is not None else "Not Enough data available!")
print("🍀 State data shape for observations:", state.shape if state is not None else "Not Enough data available!")
print(f"\n{"Indicator":<15}|{"🍋 Source at index:" + str(source_index):<30}|{"🍀 State at index: " + str(obs_index):<30}", \
      f"\n{"RSI":<15}|{source_data.iloc[source_index]['RSI_norm']:<30}|{state[obs_index][-1][0]}", \
      f"\n{"MACD_line":<15}|{source_data.iloc[source_index]['MACD_line_norm']:<30}|{state[obs_index][-1][1]}",\
      f"\n{"MACD_signal":<15}|{source_data.iloc[source_index]['MACD_signal_norm']:<30}|{state[obs_index][-1][2]}", \
      f"\n{"Volume":<15}|{source_data.iloc[source_index]['Volume_norm']:<30}|{state[obs_index][-1][3]}", \
      f"\n{"Return":<15}|{source_data.iloc[-1]['Return']:<30}|{state[obs_index][-1][4]}")

print(f"\n{"🍋 Source value":<30}|{"🍀 Observation value":<30}", \
      f"\n{"🍊 Target: " + str(source_data.iloc[-1]['Target']):<30}|{"🍊 Target:" + str(metadata[-1]['target']):<30}")

The Q-Network:

In [ ]:
import torch.nn as nn

class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(5, out_channels=64, kernel_size=5),
            nn.ReLU(),
            nn.Conv1d(64, out_channels=128, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 54, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0) 
        x = x.transpose(1, 2) # Changing the shape from (1, 60, 5) to (1, 5, 60)
        return self.network(x)

3454

In [ ]:
model_path = Path(__file__).parent / "rlmodel" / "rl_dqn.pth"

device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")

q_network = QNetwork().to(device)
q_network.load_state_dict(torch.load(model_path))

with torch.no_grad():
    # since now I am using np.float32(60, 5) shape observation, I need to change it to torch.tensor(1, 60, 5) shape before passing to the network
    observation = torch.tensor(observation, dtype=torch.float32).to(device)
    q_values = q_network(observation)
    action_index = int(torch.argmax(q_values, dim=1).item())

## Plan for backtesting
1. I will implement a code to calculate the actual trends
2. I will collect the estimated trends. 

I will plot the apple price graph, and on top of that I will place markers in Green or Red colors. It will be in Green if the actual and estimated trends match, otherwise it will be Red. This will give me visual backtesting. 

## Summary
🍋 Evaluated the DQN training result


|Date (YYYY-MM-DD)|Version|Created By|  
|--|--|--|
|2026-03-15|1.0|Battogtokh Baasanjav|